# P16: сравнение форм триггера

Notebook подготовлен до границы запуска обучения: конфигурация, данные, триггеры, poisoning, модель и smoke-проверка. Длительное обучение запускается только в последней секции.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
ROOT

In [ ]:
from p16_triggers.config import load_config

config = load_config(ROOT / "configs" / "baseline.yaml")
config

## Подготовка CIFAR-10

Следующая команда только загружает данные и не обучает модель.

In [ ]:
# Выполняется один раз, если data/raw еще пуст:
# !python ../scripts/prepare_data.py --config configs/baseline.yaml

## Smoke-проверка полного конвейера

Берется небольшая подвыборка, создается poisoned dataset и выполняется один прямой проход CNN без обновления весов.

In [ ]:
import torch
from p16_triggers.data import build_cifar10_loaders
from p16_triggers.model import build_model, count_trainable_parameters

data = build_cifar10_loaders(
    config,
    seed=17,
    trigger_name="patch",
    poison_fraction=0.03,
    download=False,
    max_train_samples=512,
    max_eval_samples=256,
)
model = build_model(config.model)
images, labels, original_labels, trigger_flags, indices = next(iter(data.train))
with torch.inference_mode():
    logits = model(images)
print(
    {
        "images": tuple(images.shape),
        "logits": tuple(logits.shape),
        "parameters": count_trainable_parameters(model),
        "poisoned_examples": len(data.poison_indices),
    }
)

## Визуальный аудит на реальных изображениях

Скрипт создает `artifacts/cifar10_trigger_grid.png` и JSON с метриками заметности.

In [ ]:
# !python ../scripts/audit_triggers.py --config configs/baseline.yaml

## Стоп: следующая команда уже запускает обучение

До этого места проект полностью подготовлен. Сначала обучается чистая модель, затем poisoned-модели.

In [ ]:
# Чистая baseline-модель:
# !python ../scripts/train.py --mode clean --trigger patch --seed 17

# Пример модели с патчем и rho=3%:
# !python ../scripts/train.py --mode poisoned --trigger patch --poison-fraction 0.03 --seed 17